## Constructed-data method demonstration: CONSTRUCTED original pairs method

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are labelled separately as replays. French raw, extracted, and processed files are not bundled, and the required rights and permitted uses have not been established; this Notebook does not analyze French data.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 5: Walk-Forward Pairs Trading Research Pipeline

> **CRITICAL RESEARCH INTEGRITY NOTICE**: All results displayed herein are evaluated on the teaching **CONSTRUCTED** panel (`data_class = 'constructed'`), **NOT MARKET EVIDENCE**.

This cold-runnable notebook implements the frozen preregistered walk-forward pairs pipeline:
1. **Vendor-Agnostic Corporate Actions**: Forward-built total-return series cleanly separated from unadjusted execution prices.
2. **Strict Walk-Forward Formation**: Rolling Engle-Granger cointegration screening ($p < 0.05$) with frozen parameters.
3. **Signal Sizing & Rules**: Frozen spread z-score thresholds ($\pm 2.0$ entry, $\pm 0.5$ exit, $\pm 4.0$ stop).
4. **Full Execution Ledger**: Strict execution lag, an ex-ante opening-fill budget (5% of ADV20 known at the signal close; an uncalibrated scenario, not a realised participation guarantee), matched vs. independent partial fills, open marks before fills, ACT/360 financing, borrow fees, and dividend cash flows.
5. **Reconciliation & Attribution**: Zero-leakage invariant checks and headline strategy evaluation.
6. **Preregistration v2**: selection is EG p < 0.05 AND beta > 0 (long/short relative-value scope; frozen before any real data, after constructed results were seen).

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p05_pairs_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, hashlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
root = ROOT / "projects/project05"
for path in (root, root / "src"): sys.path.insert(0,str(path))
import p05_constructed
import p05_pairs as pp
import run_project05


## 1. Data Ingestion & Manifest Gate

The preregistration contract requires that real market evidence is processed only when `data_manifest.json` has reached the `frozen` status. When not frozen or absent, the pipeline executes safely on the labelled teaching panel from `src/p05_constructed.py`.

In [ ]:
prereg_path = root / 'configs' / 'preregistration_v2.json'
prereg_bytes = prereg_path.read_bytes()
prereg = json.loads(prereg_bytes)
prereg_digest = hashlib.sha256(prereg_bytes).hexdigest()
manifest_path = root / 'data_manifest.json'

# Always choose the offline constructed branch; real-data execution needs separate permission.
use_real = False
panel_df, rates_df, pairs, meta = run_project05.load_inputs('constructed', prereg)
print(f'CONSTRUCTED teaching panel ({meta["data_class"]}): {len(pairs)} pairs; real route SKIPPED')


## 2. Walk-Forward Simulation Execution

Executing rolling walk-forward formation (504 days) and trading (126 days) across all pairs. Both `screened` ($p < 0.05$ formation gate) and `ungated` (always trade) sleeves are simulated to expose selection effects.

In [ ]:
results = run_project05.run_all(panel_df, rates_df, pairs, prereg)
print(f'Walk-forward completed: {len(results["windows"])} rolling windows.')
print(f'Total formation tests: {len(results["formations"])}')
print(f'EG p < 0.05: {int(results["formations"]["eg_pass"].sum())}; selected (p < 0.05 and beta > 0): {int(results["formations"]["selected"].sum())}')

## 3. Walk-Forward Formations Table

Estimated formation parameters across all windows and pairs, showing Engle-Granger statistics, Benjamini-Hochberg FDR q-values, and selection flags.

In [ ]:
df_form = results['formations'][[
    'pair', 'window', 'n', 'alpha', 'beta', 'spread_sd', 'half_life_days',
    'corr_level', 'eg_stat', 'eg_pvalue', 'qvalue_bh_window', 'eg_pass', 'selected'
]].copy()
print(df_form.to_string(index=False))

## 4. Frozen Spread Dynamics & Trade Execution Fills

Visualizing the frozen formation spread z-score against signal thresholds ($\pm 2.0$ entry, $\pm 0.5$ exit) and executed order fills in the ledger for cointegrated pair `CA_CB`.

In [ ]:
target_pair = 'CA_CB'
w_target = results['windows'][2]  # Representative window 2
form_target = results['formations'][(results['formations']['pair'] == target_pair) & (results['formations']['window'] == w_target.k)].iloc[0]

P = results['P']
log_tr = np.log(pp.total_return_index(P))
sig_dates = w_target.signal_dates
s = log_tr.loc[sig_dates, form_target['y']] - form_target['alpha'] - form_target['beta'] * log_tr.loc[sig_dates, form_target['x']]
z = (s - form_target['spread_mean']) / form_target['spread_sd']

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
ax.plot(z.index, z.values, color='steelblue', linewidth=1.8, label='Spread z-score')
ax.axhline(2.0, color='crimson', linestyle='--', label='Entry (+2.0)')
ax.axhline(0.5, color='darkorange', linestyle=':', label='Exit (+0.5)')
ax.axhline(0.0, color='gray', linestyle='-', linewidth=0.8)
ax.axhline(-0.5, color='darkorange', linestyle=':', label='Exit (-0.5)')
ax.axhline(-2.0, color='crimson', linestyle='--', label='Entry (-2.0)')

orders = results['orders']
sub_o = orders[(orders['pair'] == target_pair) & (orders['gate'] == 'screened')]
if len(sub_o):
    for _, row_o in sub_o.iterrows():
        sd = pd.Timestamp(row_o['signal_date'])
        if sd in z.index:
            zv = z.loc[sd]
            color = 'green' if row_o['kind'] == 'entry' else 'purple'
            marker = '^' if (row_o['kind'] == 'entry' and zv < 0) else ('v' if row_o['kind'] == 'entry' else 'o')
            ax.scatter(pd.Timestamp(row_o['fill_date']), zv, color=color, marker=marker, s=70, zorder=5)

ax.set_title(f'Pair {target_pair} Spread Dynamics & Ledger Execution (Window {w_target.k})\n[CONSTRUCTED — not market evidence]', fontsize=11, fontweight='bold')
ax.set_xlabel('Trading Date (YYYY-MM)')
ax.set_ylabel('Spread z-score (SD)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
fig.savefig(RUN_DIR / "pairs_spread_CONSTRUCTED.png", dpi=130)
plt.show()

## 5. Ledger Reconciliation & Friction Accounting

Verifying the fundamental ledger accounting identity:
$$\text{Cash Residual} = \text{Cash} - (W_0 + \text{Interest} - \text{Borrow} + \text{Dividends} - \text{Trade Flow} - \text{Commission}) \approx 0$$
Ensuring bid-ask spread and market impact are incorporated purely in fill prices and never debited twice.

In [ ]:
reconciliation = []
for (pair, gate), led in results['ledgers'].items():
    reconciliation.append({
        'pair': pair,
        'gate': gate,
        'interest': led['totals']['interest'],
        'borrow': led['totals']['borrow'],
        'dividends': led['totals']['dividends'],
        'commission': led['totals']['commission'],
        'spread_cost': led['totals']['spread_cost'],
        'impact_cost': led['totals']['impact_cost'],
        'cash_residual': led['cash_residual']
    })
df_rec = pd.DataFrame(reconciliation)
print('LEDGER RECONCILIATION SUMMARY:')
print(df_rec[['pair', 'gate', 'interest', 'borrow', 'dividends', 'commission', 'cash_residual']].to_string(index=False))
assert (df_rec['cash_residual'].abs() < 1e-7).all(), 'Non-zero cash residual detected!'

## 6. Headline Strategy Performance Metrics

Performance summary showing total return, annualized return, volatility, excess Sharpe ratio (over daily cash credit rate), maximum drawdown, and trade counts.

In [ ]:
metrics_df = run_project05.metrics_table(results)
print('HEADLINE METRICS TABLE [CONSTRUCTED — NOT MARKET EVIDENCE]:')
cols = ['pair', 'gate', 'n_returns', 'total_return', 'ann_return', 'ann_vol', 'sharpe_excess', 'max_drawdown', 'closed_trades', 'open_trades', 'win_rate', 'mean_holding_days', 'turnover_one_sided_ann', 'cash_residual']
print(metrics_df[cols].to_string(index=False))